# CHGIS Temporal Gazetteer (TGAZ)

This notebook demonstrates the **CHGIS Temporal Gazetteer (TGAZ)** client
in `dehergne_util.py` and shows how it can help identify Chinese place
names in the Dehergne data (Jesuits in China, 1552–1800).

**Why TGAZ?** TGAZ is the read-only REST service over the *China Historical
GIS* database (Harvard & Fudan University). Unlike Wikidata, which mostly
carries the *modern* name of a place, TGAZ records the name a place bore
**during a specific period**, together with its administrative parent and
feature type. That temporal dimension is exactly what a 17th-century
itinerary needs.

**Caveat (verified live):** the API only knows Chinese characters and
pinyin. European spellings like "Macau" or "Changchow" return nothing
useful — you must query the Chinese form (澳门) or the pinyin ("Aomen").

**Endpoint note:** the official docs still point at
`maps.cga.harvard.edu/tgaz/...`, but that host is dead (GitHub Pages
404). The service moved to `chgis.hudci.org/tgaz/...`, which the helper
uses. Docs: <https://chgis.hudci.org/tgw/#usage>

## Setup

In [2]:
from dehergne_util import (
    tgaz_query,
    tgaz_get,
    tgaz_results_to_records,
    tgaz_results_to_dataframe,
)

import pandas as pd

pd.set_option("display.max_rows", 150)
pd.set_option("display.width", 160)

ImportError: cannot import name 'tgaz_query' from 'dehergne_util' (/Users/jrc/mhk-home/sources/dehergne/notebooks/dehergne_util.py)

## 1. Faceted search by name

The basic search takes a placename and returns every TGAZ record that
matches. Try the pinyin form of a well-known Chinese city.

In [ ]:
r = tgaz_query(name="Guangzhou")
df = tgaz_results_to_dataframe(r)
print(r["memo"])
df[["name", "transcription", "years", "feature_type", "parent_name", "lat", "lon"]]

Notice the **same place appears multiple times** under different year
ranges and administrative levels (prefecture `府` vs. county `县`, etc.).
This is the core value of a *temporal* gazetteer.

## 2. Filter by year — the key feature

Pass `year=` to keep only the records valid at a given moment. This is
what lets us ask "what was Macau called, and where did it sit
administratively, in 1660?"

In [ ]:
# Macau in pinyin is "Aomen". Note: "Macau"/"Macao" return nothing useful.
r = tgaz_query(name="Aomen", year=1820)
tgaz_results_to_dataframe(r)

In [ ]:
# Guangzhou prefecture as it stood around a typical Dehergne date.
r = tgaz_query(name="Guangzhou", feature_type="fu", year=1700)
tgaz_results_to_dataframe(r)

## 3. Chinese characters work directly

UTF-8 Chinese characters can be sent as-is (no URL-encoding). This is
often the most reliable way to query, because several distinct pinyin
strings can map to the same characters.

A useful first step is to search a place **without** a year filter, to see
every period in which a given name was in use. Notice how the same
characters refer to different administrative units in different eras --
and how some familiar names (like Macau 澳门) only enter the record late,
because they were not Chinese administrative units during the Dehergne
period (1552-1800).

In [ ]:
# Search each place across ALL years (no year filter).
for place in ["杭州", "北京", "澳门"]:
    r = tgaz_query(name=place)
    print(f"\n{place}: {r['count of total results']} record(s)")
    display(tgaz_results_to_dataframe(r).head(5))

## 4. Other facets: feature type, parent, source

The remaining facets refine a name search rather than stand alone.
The live server returns an empty body for any query that does **not**
include a name, so `feature_type`, `year`, `source` and `parent` are
useful as filters on a name, not as browse dimensions.


In [ ]:
# "Guangzhou" appears at many administrative levels and many periods.
# Narrow to prefecture (fu) units valid in 1700.
r = tgaz_query(name="Guangzhou", feature_type="fu", year=1700)
tgaz_results_to_dataframe(r)

## 5. Canonical record by ID

Once you have a `sys_id` (the `hvd_...` identifier) from a search, you
can fetch its full canonical record. This endpoint returns a **richer**
structure: traditional + simplified Chinese spellings, pinyin, feature
type with English translation, temporal span rules, and coordinates.

In [ ]:
# hvd_33047 is Guangzhou Fu (prefecture).
record = tgaz_get("hvd_33047")
record

Pull out the parts you usually care about:

In [ ]:
spellings = [s.get("written form") for s in record.get("spellings", [])]
ft = record.get("feature_type", {})
temp = record.get("temporal", {})
spatial = record.get("spatial", {})
print("spellings      :", spellings)
print("feature type   :", ft.get("transcription"), "|", ft.get("English"))
print("temporal span  :", temp.get("begin"), "→", temp.get("end"))
print("coordinates    :", spatial.get("latitude"), spatial.get("longitude"))

## 6. Mapping Dehergne place names to TGAZ

Dehergne records places in European (mostly Portuguese/Latin) spellings.
TGAZ only knows Chinese characters and pinyin, so we need a lookup from
the European spelling to the **historical** Chinese name -- which is not
always the modern one. Two findings from the raw searches above shape
this table:

* **Macau (澳门)** -- TGAZ has no record before 1820. Macau was a
  Portuguese settlement, not a Chinese administrative unit, during the
  Jesuit period, so it is genuinely absent. Use Wikidata for Macau.
* **Peking** -- the Qing capital is recorded under its prefecture name
  **顺天府 (Shuntian Fu)**, not 北京.
* **Nanking** -- recorded as **江宁府 (Jiangning Fu)** in the Qing, not 南京.

This is precisely the historical-name disambiguation that makes TGAZ
worth the extra step.

In [ ]:
# European spelling -> (Chinese characters to query, a Dehergne-era year).
# Note the historical names, not the modern ones, for Peking and Nanking.
DEHERGNE_TO_TGAZ = {
    "Macau":    ("澳门",   1820),  # earliest TGAZ record (post-Jesuit)
    "Canton":   ("广州",   1700),  # Guangzhou Fu
    "Hangchow": ("杭州",   1700),  # Hangzhou Fu
    "Peking":   ("顺天府", 1700),  # Shuntian Fu (the Qing capital)
    "Nanking":  ("江宁府", 1700),  # Jiangning Fu (Qing Nanjing)
}

rows = []
for euro, (chars, yr) in DEHERGNE_TO_TGAZ.items():
    r = tgaz_query(name=chars, year=yr)
    for rec in tgaz_results_to_records(r):
        rows.append({
            "dehergne": euro,
            "tgaz_name": rec["name"],
            "transcription": rec["transcription"],
            "years": rec["years"],
            "feature_type": rec["feature_type"],
            "parent_name": rec["parent_name"],
            "lat": rec["lat"],
            "lon": rec["lon"],
            "sys_id": rec["sys_id"],
        })

matches = pd.DataFrame(rows)
matches

### Cross-checking with Wikidata

The coordinates from TGAZ can be compared against the Wikidata
coordinates already stored in
`../inferences/wikidata-references/locations_wikidata_info.xlsx`. When
the two agree, confidence in the identification is high; when they
disagree, the record needs manual review (the place may have been
relocated, or the European name may be ambiguous).

In [ ]:
from dehergne_util import locations_wikidata_info_file

try:
    wd = pd.read_excel(locations_wikidata_info_file)
    print("Wikidata reference file:", locations_wikidata_info_file)
    print("rows:", len(wd), "| columns:", list(wd.columns)[:8])
except FileNotFoundError:
    print("Wikidata reference file not found at", locations_wikidata_info_file)
    print("(skipping the cross-check; generate it from location-analysis-new.ipynb)")

## 7. Parameter validation

The helper validates inputs so bad queries fail fast with a clear
message.

In [ ]:
# Years outside the documented range -222..1911 are rejected.
try:
    tgaz_query(name="Aomen", year=2000)
except ValueError as e:
    print("rejected ->", e)

# At least one search facet is required.
try:
    tgaz_query()
except ValueError as e:
    print("rejected ->", e)

## Summary

| Helper | Purpose |
|---|---|
| `tgaz_query(name, year, feature_type, source, parent)` | faceted search |
| `tgaz_get(sys_id)` | full canonical record by `hvd_` id |
| `tgaz_results_to_records(response)` | clean dict list from a search |
| `tgaz_results_to_dataframe(response)` | same, as a pandas DataFrame |

**When to reach for TGAZ vs Wikidata**

* **Wikidata** — best for the modern identity of a place, multilingual
  labels, and linking out to other knowledge graphs.
* **TGAZ** — best for the *historical* name and administrative position
  of a Chinese place *at a specific year*, which is exactly the question
  a 1552–1800 itinerary raises.

They are complementary: use Wikidata to anchor the modern entity, and
TGAZ to disambiguate which historical instance a Dehergne entry refers
to.